# 05 — Retrieving Real Biological Data from NCBI with Biopython

**From public database to reproducible sequence analysis**

In this notebook, we will retrieve **real nucleotide records from NCBI Nucleotide/GenBank** using Biopython's `Bio.Entrez` API, save the downloaded records locally, parse FASTA and GenBank formats, and perform basic sequence-quality checks.

Unlike the earlier notebooks, the main data in this notebook are **not invented examples**: the notebook downloads them from NCBI when you run the relevant cells.

> **Requirements:** internet access and Biopython. NCBI requires a valid contact email for Entrez requests. Replace the email placeholder in the setup cell with your own email address. Do not send frequent or unnecessary requests.


## Learning objectives

By the end, you will be able to:

- Configure `Bio.Entrez` responsibly.
- Fetch a known accession from NCBI as FASTA and GenBank.
- Save original downloaded files so the analysis can be reproduced without downloading again.
- Parse records with `Bio.SeqIO`.
- Search NCBI Nucleotide and fetch a small set of matching records.
- Summarize sequence length, GC percentage, ambiguous bases, and record metadata.
- Build a small, reusable download-and-analysis pipeline.

**Biological example:** human **TP53 mRNA**, accession `NM_000546.6`. Accession versions can change over time, so always keep the accession and download date with your results.


In [1]:
# Install Biopython if needed (run in a notebook environment)
# %pip install biopython

import os
import csv
import time
from pathlib import Path
from datetime import datetime, timezone

from Bio import Entrez, SeqIO
from Bio.SeqUtils import gc_fraction

print("Biopython is ready.")


Biopython is ready.


## 1. Configure access to NCBI

NCBI asks programmatic users to identify themselves with an email address. Use your real contact email below. An API key is optional for this small exercise.

The Entrez utilities have rate limits. Biopython includes retry behavior and rate-aware handling, but you should still avoid unnecessary repeated requests. For repeated analyses, download once and work from the saved local files.

Official references:
- [NCBI Entrez Programming Utilities](https://www.ncbi.nlm.nih.gov/books/NBK25501/)
- [Biopython Entrez tutorial](https://biopython.org/docs/latest/Tutorial/chapter_entrez.html)


In [3]:
# IMPORTANT: replace this with your own contact email before making requests.
Entrez.email = "YOUR_EMAIL@example.com"
Entrez.tool = "BiopythonNCBITrainingNotebook"

# Optional: if you have an NCBI API key, uncomment and set it securely.
# Entrez.api_key = os.environ.get("NCBI_API_KEY")

if Entrez.email == "mobinZ2014@gmail.com":
    raise ValueError("Edit Entrez.email above and enter your own contact email before querying NCBI.")

DATA_DIR = Path("data_ncbi")
DATA_DIR.mkdir(exist_ok=True)

print("Output directory:", DATA_DIR.resolve())


Output directory: C:\Users\mobin\Downloads\data_ncbi


## 2. Download one real record by accession

`Entrez.efetch()` retrieves a record from NCBI. We request the same accession in two formats:

- **FASTA** — sequence plus a short header.
- **GenBank** — sequence plus annotations and feature information.

The `if not path.exists()` check prevents repeated downloads when the files are already present. Delete the files only if you intentionally want to fetch them again.


In [4]:
accession = "NM_000546.6"  # Human TP53 transcript; versioned RefSeq accession
fasta_path = DATA_DIR / f"{accession}.fasta"
genbank_path = DATA_DIR / f"{accession}.gb"

def download_ncbi_record(accession: str, rettype: str, output_path: Path) -> Path:
    """Download one NCBI nucleotide record and save it locally if not already present."""
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"Using existing file: {output_path}")
        return output_path

    with Entrez.efetch(
        db="nucleotide",
        id=accession,
        rettype=rettype,
        retmode="text"
    ) as handle:
        content = handle.read()

    if not content.strip():
        raise RuntimeError(f"NCBI returned an empty response for {accession} ({rettype}).")

    output_path.write_text(content, encoding="utf-8")
    print(f"Downloaded and saved: {output_path} ({output_path.stat().st_size:,} bytes)")
    return output_path

download_ncbi_record(accession, "fasta", fasta_path)
time.sleep(0.4)  # Be considerate to NCBI
download_ncbi_record(accession, "gb", genbank_path)


Downloaded and saved: data_ncbi\NM_000546.6.fasta (2,666 bytes)
Downloaded and saved: data_ncbi\NM_000546.6.gb (38,506 bytes)


WindowsPath('data_ncbi/NM_000546.6.gb')

## 3. Parse the downloaded FASTA with `SeqIO`

FASTA is ideal when you primarily need the sequence. Use `SeqIO.read()` when exactly one record is expected; it raises an error if the file contains zero or multiple records.


In [5]:
tp53_fasta_record = SeqIO.read(fasta_path, "fasta")

print("Record ID:", tp53_fasta_record.id)
print("Description:", tp53_fasta_record.description)
print("Sequence length:", len(tp53_fasta_record.seq), "nt")
print("First 60 bases:", tp53_fasta_record.seq[:60])


Record ID: NM_000546.6
Description: NM_000546.6 Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA
Sequence length: 2512 nt
First 60 bases: CTCAAAAGTCTAGAGCCACCGTCCAGGGAGCAGGTAGCTGCTGGGCTCCGGGGACACTTT


## 4. Parse GenBank annotations and features

GenBank records preserve additional information such as organism, accession, references, and annotated features. The exact feature set can change as NCBI updates records.


In [6]:
tp53_gb_record = SeqIO.read(genbank_path, "genbank")

print("ID:", tp53_gb_record.id)
print("Name:", tp53_gb_record.name)
print("Description:", tp53_gb_record.description)
print("Organism:", tp53_gb_record.annotations.get("organism", "Not provided"))
print("Molecule type:", tp53_gb_record.annotations.get("molecule_type", "Not provided"))
print("Number of annotated features:", len(tp53_gb_record.features))

# Inspect a few feature types without printing a huge record.
feature_types = {}
for feature in tp53_gb_record.features:
    feature_types[feature.type] = feature_types.get(feature.type, 0) + 1
feature_types


ID: NM_000546.6
Name: NM_000546
Description: Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA
Organism: Homo sapiens
Molecule type: mRNA
Number of annotated features: 70


{'source': 1,
 'gene': 1,
 'exon': 11,
 'misc_feature': 54,
 'CDS': 1,
 'regulatory': 1,
 'polyA_site': 1}

## 5. Basic sequence QC

This section calculates simple descriptive statistics. These checks are not a substitute for biological validation, but they are useful for catching unexpected or malformed sequence data.

For nucleotide sequences, we will calculate length, GC percentage, and the number of ambiguous/non-ACGT symbols.


In [7]:
def sequence_qc(record):
    seq = str(record.seq).upper()
    canonical = set("ACGT")
    ambiguous_count = sum(base not in canonical for base in seq)
    gc_percent = 100 * gc_fraction(seq) if seq else 0.0

    return {
        "record_id": record.id,
        "description": record.description,
        "length_nt": len(seq),
        "gc_percent": round(gc_percent, 3),
        "ambiguous_or_non_acgt": ambiguous_count,
    }

qc = sequence_qc(tp53_fasta_record)
qc


{'record_id': 'NM_000546.6',
 'description': 'NM_000546.6 Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA',
 'length_nt': 2512,
 'gc_percent': 53.384,
 'ambiguous_or_non_acgt': 0}

## 6. Search NCBI for real records

Now use `Entrez.esearch()` to find a small set of nucleotide records. This example searches for human BRCA1 mRNA records in RefSeq. Search results can change over time, so the returned accessions are discovered live rather than hard-coded.

We deliberately limit the number of records to keep this exercise small. For large searches, use NCBI's history mechanism and fetch records in batches.


In [8]:
search_term = 'BRCA1[Gene] AND Homo sapiens[Organism] AND biomol_mrna[PROP] AND srcdb_refseq[PROP]'

with Entrez.esearch(
    db="nucleotide",
    term=search_term,
    retmax=5,
    sort="relevance"
) as handle:
    search_results = Entrez.read(handle)

print("Total matching records reported by NCBI:", search_results["Count"])
accessions = search_results["IdList"]
print("Accessions returned:", accessions)


Total matching records reported by NCBI: 368
Accessions returned: ['1732746264', '1677529721', '1677501432', '1677500831', '2255456859']


## 7. Fetch the search results and save a multi-record FASTA

`Entrez.efetch()` can fetch multiple accessions in one request. We save the raw response to disk and then parse it using `SeqIO.parse()`, which is appropriate for files containing multiple records.


In [9]:
search_fasta_path = DATA_DIR / "human_BRCA1_refseq_mrna_top5.fasta"

if not accessions:
    print("No records returned. Try a broader search term or check your internet connection.")
else:
    if not search_fasta_path.exists() or search_fasta_path.stat().st_size == 0:
        with Entrez.efetch(
            db="nucleotide",
            id=",".join(accessions),
            rettype="fasta",
            retmode="text"
        ) as handle:
            fasta_text = handle.read()
        if not fasta_text.strip():
            raise RuntimeError("NCBI returned an empty FASTA response for the search results.")
        search_fasta_path.write_text(fasta_text, encoding="utf-8")
        print("Saved:", search_fasta_path)
    else:
        print("Using existing file:", search_fasta_path)

brca_records = list(SeqIO.parse(search_fasta_path, "fasta"))
print("Parsed records:", len(brca_records))
for record in brca_records:
    print(record.id, len(record.seq), "nt", "|", record.description)


Saved: data_ncbi\human_BRCA1_refseq_mrna_top5.fasta
Parsed records: 5
NM_007294.4 7088 nt | NM_007294.4 Homo sapiens BRCA1 DNA repair associated (BRCA1), transcript variant 1, mRNA
NM_007300.4 7151 nt | NM_007300.4 Homo sapiens BRCA1 DNA repair associated (BRCA1), transcript variant 2, mRNA
NM_007299.4 3696 nt | NM_007299.4 Homo sapiens BRCA1 DNA repair associated (BRCA1), transcript variant 5, mRNA
NM_007297.4 7028 nt | NM_007297.4 Homo sapiens BRCA1 DNA repair associated (BRCA1), transcript variant 3, mRNA
NM_001408467.1 3716 nt | NM_001408467.1 Homo sapiens BRCA1 DNA repair associated (BRCA1), transcript variant 326, mRNA


## 8. Create a tabular report

We will create a CSV report from the downloaded BRCA1 records. CSV files can be opened in spreadsheet software or loaded into pandas for downstream analysis.


In [10]:
report_path = DATA_DIR / "human_BRCA1_refseq_mrna_qc.csv"
report_rows = [sequence_qc(record) for record in brca_records]

if report_rows:
    with report_path.open("w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=list(report_rows[0].keys()))
        writer.writeheader()
        writer.writerows(report_rows)

    print("Saved report:", report_path.resolve())
    report_rows
else:
    print("No records available to report.")


Saved report: C:\Users\mobin\Downloads\data_ncbi\human_BRCA1_refseq_mrna_qc.csv


## 9. Reproducibility: record the download metadata

For a scientific workflow, keep a small metadata file with the database, accessions, query, retrieval time, and filenames. NCBI records can be updated, and search rankings can change, so this information helps you document exactly what was retrieved.


In [13]:
import json

metadata = {
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "database": "NCBI Nucleotide",
    "single_record_accession": accession,
    "search_database": "nucleotide",
    "search_term": search_term,
    "search_accessions": accessions,
    "files": [
        str(fasta_path),
        str(genbank_path),
        str(search_fasta_path),
        str(report_path),
    ],
}

metadata_path = DATA_DIR / "download_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved metadata:", metadata_path.resolve())


Saved metadata: C:\Users\mobin\Downloads\data_ncbi\download_metadata.json


## 10. Mini-project: make the downloader reusable

The function below searches a user-provided query, fetches a limited number of FASTA records, and returns parsed `SeqRecord` objects. It intentionally has a conservative limit and uses one batch request.

Before adapting it for a real research project, consider query validation, duplicate handling, pagination/history for large result sets, retries, checksums, and saving the exact search strategy.


In [14]:
def search_and_download_nucleotide_fasta(
    query: str,
    output_file: str | Path,
    max_records: int = 10
):
    """Search NCBI Nucleotide and download up to max_records results as FASTA."""
    if not query.strip():
        raise ValueError("query must not be empty")
    if not 1 <= max_records <= 100:
        raise ValueError("For this teaching function, max_records must be between 1 and 100")

    with Entrez.esearch(
        db="nucleotide",
        term=query,
        retmax=max_records,
        sort="relevance"
    ) as handle:
        results = Entrez.read(handle)

    ids = results["IdList"]
    if not ids:
        print("No records found.")
        return []

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with Entrez.efetch(
        db="nucleotide",
        id=",".join(ids),
        rettype="fasta",
        retmode="text"
    ) as handle:
        fasta_text = handle.read()

    if not fasta_text.strip():
        raise RuntimeError("NCBI returned an empty response.")

    output_file.write_text(fasta_text, encoding="utf-8")
    records = list(SeqIO.parse(output_file, "fasta"))
    print(f"NCBI reported {results['Count']} matches; downloaded {len(records)} records.")
    print(f"Saved to: {output_file.resolve()}")
    return records

# Example usage (makes a live NCBI request):
# records = search_and_download_nucleotide_fasta(
#     "COI[Gene] AND Drosophila melanogaster[Organism]",
#     DATA_DIR / "drosophila_COI.fasta",
#     max_records=5
# )


## Exercises

1. Change the accession to another versioned RefSeq nucleotide accession and compare the length and annotations.
2. Search for a gene in another organism and save five records as FASTA.
3. Extend `sequence_qc()` to report the counts of A, C, G, T, and N.
4. Compare FASTA and GenBank records for the same accession. Which metadata are absent from FASTA?
5. Add a SHA-256 checksum for each downloaded file to the metadata JSON.
6. Explain why a reproducible report should preserve the query, retrieval date, accession version, and original downloaded file.

## Common issues

- **HTTP/network error:** confirm internet access and rerun later; remote services can be temporarily unavailable.
- **Email placeholder error:** replace `YOUR_EMAIL@example.com` with your own email.
- **No search results:** broaden the query or verify the gene/organism terms in the NCBI web interface.
- **Different results on a later run:** NCBI records and search rankings can change; use accession versions and retain downloaded files.
- **Too many requests:** limit query size, reuse local files, and follow the [NCBI usage guidance](https://www.ncbi.nlm.nih.gov/books/NBK25497/).

## Quick reference

| Task | Biopython tool |
|---|---|
| Search NCBI | `Entrez.esearch()` |
| Fetch records | `Entrez.efetch()` |
| Parse one record | `SeqIO.read()` |
| Parse multiple records | `SeqIO.parse()` |
| Save a record file | `SeqIO.write()` or write the raw response |
| Track annotations | `SeqRecord.annotations`, `SeqRecord.features` |

**Next notebook suggestion:** sequence alignment and similarity search — using real sequences downloaded here as input, then exploring pairwise alignment and BLAST concepts.
